# Technical Indicators Notebook

In this notebook, we explore commonly used **technical indicators** for analysing financial markets. Technical indicators transform raw price and volume data into structured signals that help traders assess trend direction, momentum, volatility, and potential entry or exit points.

We begin by installing two key libraries:

- **yfinance** – Used to download historical market data (prices, volume, etc.) directly from Yahoo Finance.
- **ta** – A Python technical analysis library that provides ready-to-use implementations of popular indicators such as RSI, MACD, Bollinger Bands, and moving averages.

In [1]:
# Uncomment the following lines to install required packages
#!pip install yfinance
#!pip install ta

Import relevant libraries

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import math
from ta.utils import dropna
from ta.volatility import BollingerBands
from ta.trend import ADXIndicator
from ta.volatility import AverageTrueRange
from ta.trend import SMAIndicator
from ta.momentum import RSIIndicator
from ta.volume import VolumeWeightedAveragePrice

Since we are just exploring the technical indicators, we can simply reference one stock

In [3]:
apple_symbol = "AAPL"
start = "2022-09-01"
end = "2023-09-06"

Retrieve the data

In [4]:
data = yf.Ticker(apple_symbol).history(start=start, end=end)

### Common Technical Indicators

Though the list is not exhaustive, we can manually implement some commonly used indicators in trading.

---

**1. Relative Strength Index (RSI)**

The RSI measures momentum by comparing recent gains to recent losses over a rolling window.

Let:

- $U_t$ = positive price changes  
- $D_t$ = absolute value of negative price changes  
- $n$ = lookback window  

Average gains and losses:

$$
\text{AvgGain} = \frac{1}{n} \sum_{i=1}^{n} U_i
$$

$$
\text{AvgLoss} = \frac{1}{n} \sum_{i=1}^{n} D_i
$$

Relative Strength:

$$
RS = \frac{\text{AvgGain}}{\text{AvgLoss}}
$$

RSI:

$$
RSI = 100 - \frac{100}{1 + RS}
$$


**2. Simple Moving Average (SMA)**

The SMA smooths price data by taking the arithmetic mean over a rolling window:

$$
SMA_t = \frac{1}{n} \sum_{i=0}^{n-1} P_{t-i}
$$

Where:

- $P_t$ = closing price at time $t$  
- $n$ = window length  


**3. Average Directional Index (ADX)**

ADX measures trend strength (not direction).

First compute directional movements:

$$
+DM = High_t - High_{t-1}
$$

$$
-DM = Low_{t-1} - Low_t
$$

Directional Indicators:

$$
+DI = 100 \times \frac{Smoothed(+DM)}{ATR}
$$

$$
-DI = 100 \times \frac{Smoothed(-DM)}{ATR}
$$

ADX:

$$
ADX = 100 \times \text{Smoothed} \left( \frac{|+DI - -DI|}{+DI + -DI} \right)
$$

- Higher ADX → stronger trend  
- Lower ADX → weak or ranging market  


**4. Average True Range (ATR)**

ATR measures volatility.

True Range (TR):

$$
TR_t = \max
\begin{cases}
High_t - Low_t \\
|High_t - Close_{t-1}| \\
|Low_t - Close_{t-1}|
\end{cases}
$$

ATR:

$$
ATR = \text{Smoothed Moving Average of } TR
$$


**5. Volume Weighted Average Price (VWAP)**

VWAP represents the average price weighted by trading volume:

$$
VWAP_t = \frac{\sum (Price_t \times Volume_t)}{\sum Volume_t}
$$

Where price is typically:

$$
Price_t = \frac{High_t + Low_t + Close_t}{3}
$$

VWAP is commonly used as an execution benchmark and intraday fair value reference.

In [5]:
# Calculate Technical Indicators (RSI, SMA and ATR) for Apple
def RSI(data, window):
  rsi = RSIIndicator(close = data['Close'], window = window, fillna = False)
  RSI = rsi.rsi()
  return round(RSI, 1)

def MovingAverage(data, window):
  sma = SMAIndicator(close = data['Close'], window = window, fillna = False)
  MA = sma.sma_indicator()
  return round(MA, 1)

def Trend(data, window):
  adx = ADXIndicator(high = data['High'], low = data['Low'], close = data['Close'], window = window, fillna = False)
  DI_pos = adx.adx_pos()
  DI_neg = adx.adx_neg()
  ADX = adx.adx()
  return round(ADX, 1), round(DI_pos, 1), round(DI_neg, 1)

def ATR(data, window):
  atr = AverageTrueRange(high = data['High'], low = data['Low'], close = data['Close'], window = window, fillna = False)
  ATR = atr.average_true_range()
  return round(ATR, 1)

def VWAP(data, window):
  vwap = VolumeWeightedAveragePrice(high = data['High'], low = data['Low'], close = data['Close'], volume = data['Volume'], window = window, fillna = False)
  VWAP = vwap.volume_weighted_average_price()
  return round(VWAP, 1)

The following indicators are centered around Apple Stock (AAPL)

Relative Strength Index (RSI)

In [6]:
rsi_data = RSI(data, 10)
rsi_data.tail()

Date
2023-08-29 00:00:00-04:00    58.6
2023-08-30 00:00:00-04:00    64.7
2023-08-31 00:00:00-04:00    65.1
2023-09-01 00:00:00-04:00    67.7
2023-09-05 00:00:00-04:00    68.1
Name: rsi, dtype: float64

Moving Average

In [7]:
ma_data = MovingAverage(data, 10)
ma_data.tail()

Date
2023-08-29 00:00:00-04:00    175.8
2023-08-30 00:00:00-04:00    176.9
2023-08-31 00:00:00-04:00    178.3
2023-09-01 00:00:00-04:00    179.7
2023-09-05 00:00:00-04:00    181.1
Name: sma_10, dtype: float64

Trend

In [8]:
ADX, DI_pos, DI_neg = Trend(data, 10)

trend_df = pd.DataFrame(columns = ['ADX', 'DI Positive', 'DI Negative'], index = pd.DataFrame(ADX).index)
trend_df['ADX'] = ADX
trend_df['DI Positive'] = DI_pos
trend_df['DI Negative'] = DI_neg

trend_df.tail()

,ADX,DI Positive,DI Negative
Date,,,
2023-08-29 00:00:00-04:00,33.8,32.0,22.3
2023-08-30 00:00:00-04:00,33.4,37.1,19.9
2023-08-31 00:00:00-04:00,33.6,39.1,18.9
2023-09-01 00:00:00-04:00,34.0,39.1,17.6
2023-09-05 00:00:00-04:00,33.8,36.1,18.4


Average True Range

In [9]:
atr_data = ATR(data, 10)
atr_data.tail()

Date
2023-08-29 00:00:00-04:00    3.4
2023-08-30 00:00:00-04:00    3.4
2023-08-31 00:00:00-04:00    3.2
2023-09-01 00:00:00-04:00    3.1
2023-09-05 00:00:00-04:00    3.0
Name: atr, dtype: float64

Volume Weighted Average Price

In [10]:
vwap_data = VWAP(data, 10)
vwap_data.tail()

Date
2023-08-29 00:00:00-04:00    175.5
2023-08-30 00:00:00-04:00    176.6
2023-08-31 00:00:00-04:00    178.1
2023-09-01 00:00:00-04:00    179.7
2023-09-05 00:00:00-04:00    180.9
Name: vwap_10, dtype: float64